# DVC 실습 예제: `data/mydata.csv` 버전 관리

이 노트북은 **임의의 CSV 데이터셋을 생성하고, DVC로 추적한 뒤, 데이터를 다시 수정하여 새로운 데이터 버전으로 관리하는 전체 흐름**을 보여줍니다.

## 이 노트북에서 확인할 수 있는 항목
- GitHub 저장소 링크 제출 가능 여부 확인
- `README.md` 존재 여부 확인
- 최소 2개 이상의 의미 있는 commit 생성
- `.dvc` 파일 존재 확인
- `dvc remote list` 결과 확인
- 데이터 변경 전/후 차이 설명
- `Git만으로 부족한 이유` 서술

## 실행 전 전제
- 이 노트북은 **Git 저장소 루트 또는 그 하위**에서 실행한다고 가정합니다.
- `git`, `dvc`, `python`, `pandas`가 설치되어 있어야 합니다.
- Git 커밋을 만들 예정이므로 `git config user.name`, `git config user.email`이 설정되어 있어야 합니다.
- 이 노트북은 **`data/mydata.csv`** 를 새로 만들어 DVC로 관리합니다.


In [1]:
from pathlib import Path
import os
import shlex
import subprocess
import textwrap
import pandas as pd
from IPython.display import display, Markdown

def run(cmd, check=True):
    if isinstance(cmd, str):
        printable = cmd
        result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    else:
        printable = " ".join(shlex.quote(str(x)) for x in cmd)
        result = subprocess.run(cmd, text=True, capture_output=True)
    print(f"$ {printable}")
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed: {printable}")
    return result

repo_root = Path(run(["git", "rev-parse", "--show-toplevel"]).stdout.strip())
os.chdir(repo_root)

DATA_DIR = repo_root / "data"
CSV_FILE = DATA_DIR / "mydata.csv"
DVC_FILE = DATA_DIR / "mydata.csv.dvc"
REMOTE_NAME = "practice_local"
REMOTE_DIR = Path.home() / "dvcstore" / "mlops_assignment_practice"

print("Repository root:", repo_root)
print("Working directory:", Path.cwd())


$ git rev-parse --show-toplevel
/Users/chaeyooneom/projects/mlops_assignment/ajou-mlops-2026-1nd-semester

Repository root: /Users/chaeyooneom/projects/mlops_assignment/ajou-mlops-2026-1nd-semester
Working directory: /Users/chaeyooneom/projects/mlops_assignment/ajou-mlops-2026-1nd-semester


## 1) 저장소/환경 상태 확인

아래 셀은 제출 전에 확인해야 할 정보들을 먼저 보여줍니다.

- `README.md` 존재 여부
- 현재 Git 브랜치
- GitHub 원격 저장소 링크
- DVC 버전
- 현재 설정된 DVC remote 목록


In [2]:
print("README.md exists:", (repo_root / "README.md").exists())
run(["git", "branch", "--show-current"], check=False)
run(["git", "remote", "-v"], check=False)
run(["git", "remote", "get-url", "origin"], check=False)
run(["dvc", "version"], check=False)
run(["dvc", "remote", "list"], check=False)


README.md exists: True
$ git branch --show-current
main

$ git remote -v
origin	https://github.com/ChaeYoonUm/mlops_assignment.git (fetch)
origin	https://github.com/ChaeYoonUm/mlops_assignment.git (push)

$ git remote get-url origin
https://github.com/ChaeYoonUm/mlops_assignment.git

$ dvc version
DVC version: 3.67.0 (pip)
-------------------------
Platform: Python 3.12.9 on macOS-26.4-arm64-arm-64bit
Subprojects:
	dvc_data = 3.18.3
	dvc_objects = 5.2.0
	dvc_render = 1.0.2
	dvc_task = 0.40.2
	scmrepo = 3.6.1
Supports:
	http (aiohttp = 3.13.3, aiohttp-retry = 2.9.1),
	https (aiohttp = 3.13.3, aiohttp-retry = 2.9.1)
Config:
	Global: /Users/chaeyooneom/Library/Application Support/dvc
	System: /Library/Application Support/dvc
Cache types: reflink, hardlink, symlink
Cache directory: apfs on /dev/disk3s1s1
Caches: local
Remotes: local
Workspace directory: apfs on /dev/disk3s1s1
Repo: dvc, git
Repo.site_cache_dir: /Library/Caches/dvc/repo/d38ee0d75602d03accfa5240c0c6d8b5

$ dvc remote list
lo

CompletedProcess(args=['dvc', 'remote', 'list'], returncode=0, stdout='localstore      /Users/chaeyooneom/dvcstore/mlops_assignment    (default)\n', stderr='')

## 2) 실습용 local DVC remote 준비

실무에서는 DVC remote를 공용 스토리지(NAS, SSH 서버, S3, 공유 마운트)로 두는 경우가 많습니다.  
이 실습에서는 **내 홈 디렉터리 밖의 별도 경로**가 아니라, **레포 밖의 홈 디렉터리 아래 local 폴더**를 remote처럼 사용합니다.

- remote 이름: `practice_local`
- remote 경로: `~/dvcstore/mlops_assignment_practice`

이미 같은 이름의 remote가 있으면 재사용하고, 없으면 새로 등록합니다.


In [3]:
REMOTE_DIR.mkdir(parents=True, exist_ok=True)
remote_list_before = run(["dvc", "remote", "list"], check=False).stdout

if REMOTE_NAME not in remote_list_before:
    run(["dvc", "remote", "add", REMOTE_NAME, str(REMOTE_DIR)], check=True)
    print(f"Added DVC remote: {REMOTE_NAME} -> {REMOTE_DIR}")
else:
    print(f"DVC remote '{REMOTE_NAME}' already exists. Reusing it.")

print("\nCurrent DVC remotes:")
run(["dvc", "remote", "list"], check=False)


$ dvc remote list
localstore      /Users/chaeyooneom/dvcstore/mlops_assignment    (default)

$ dvc remote add practice_local /Users/chaeyooneom/dvcstore/mlops_assignment_practice
Added DVC remote: practice_local -> /Users/chaeyooneom/dvcstore/mlops_assignment_practice

Current DVC remotes:
$ dvc remote list
localstore      /Users/chaeyooneom/dvcstore/mlops_assignment    (default)
practice_local  /Users/chaeyooneom/dvcstore/mlops_assignment_practice



CompletedProcess(args=['dvc', 'remote', 'list'], returncode=0, stdout='localstore      /Users/chaeyooneom/dvcstore/mlops_assignment    (default)\npractice_local  /Users/chaeyooneom/dvcstore/mlops_assignment_practice\n', stderr='')

## 3) 실습용 CSV 생성: `data/mydata.csv`

여기서는 간단한 고객 데이터 형태의 CSV를 하나 만듭니다.


In [4]:
DATA_DIR.mkdir(parents=True, exist_ok=True)

df_v1 = pd.DataFrame(
    [
        {"user_id": 101, "name": "Alice", "age": 25, "city": "Suwon", "purchase_amount": 19.5},
        {"user_id": 102, "name": "Bob", "age": 29, "city": "Seoul", "purchase_amount": 42.0},
        {"user_id": 103, "name": "Charlie", "age": 31, "city": "Busan", "purchase_amount": 15.0},
    ]
)

df_v1.to_csv(CSV_FILE, index=False)
print(f"Created: {CSV_FILE}")
display(df_v1)


Created: /Users/chaeyooneom/projects/mlops_assignment/ajou-mlops-2026-1nd-semester/data/mydata.csv


,user_id,name,age,city,purchase_amount
0,101,Alice,25,Suwon,19.5
1,102,Bob,29,Seoul,42.0
2,103,Charlie,31,Busan,15.0


## 4) Git에서 직접 추적 중이면 해제하고, DVC로 1차 추적

한 파일은 보통 **Git이 직접 추적**하거나 **DVC가 추적**하거나 둘 중 하나로 관리하는 것이 일반적입니다.  
따라서 `data/mydata.csv`가 Git에 직접 올라가 있다면 먼저 인덱스에서 제거한 뒤 DVC로 넘깁니다.


In [5]:
tracked = run(["git", "ls-files", "--error-unmatch", str(CSV_FILE.relative_to(repo_root))], check=False)
if tracked.returncode == 0:
    print("The file is currently tracked by Git directly. Removing from Git index first...")
    run(["git", "rm", "--cached", str(CSV_FILE.relative_to(repo_root))], check=True)
else:
    print("The file is not directly tracked by Git. Good to proceed with DVC.")

run(["dvc", "add", str(CSV_FILE.relative_to(repo_root))], check=True)

gitignore_path = DATA_DIR / ".gitignore"
to_add = []
for p in [gitignore_path, DVC_FILE]:
    if p.exists():
        to_add.append(str(p.relative_to(repo_root)))

# DVC config changes can also be part of the commit if needed.
if (repo_root / ".dvc").exists():
    to_add.append(".dvc")

print("Staging these files in Git:")
print(to_add)
run(["git", "add", *to_add], check=True)
run(["git", "status", "--short"], check=False)


$ git ls-files --error-unmatch data/mydata.csv
error: pathspec 'data/mydata.csv' did not match any file(s) known to git
Did you forget to 'git add'?

The file is not directly tracked by Git. Good to proceed with DVC.
$ dvc add data/mydata.csv

To track the changes with git, run:

	git add data/.gitignore data/mydata.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true

⠋ Checking graph


Staging these files in Git:
['data/.gitignore', 'data/mydata.csv.dvc', '.dvc']
$ git add data/.gitignore data/mydata.csv.dvc .dvc
$ git status --short
M  .dvc/config
M  data/.gitignore
A  data/mydata.csv.dvc
?? data_versioning_pracitce/



CompletedProcess(args=['git', 'status', '--short'], returncode=0, stdout='M  .dvc/config\nM  data/.gitignore\nA  data/mydata.csv.dvc\n?? data_versioning_pracitce/\n', stderr='')

## 5) 첫 번째 의미 있는 commit 생성

이 commit은 다음 의미를 가집니다.

- `data/mydata.csv`를 생성했다
- 이 파일을 Git이 아니라 DVC로 관리하도록 전환했다
- `.dvc` 메타파일과 `.gitignore` 규칙을 저장소에 반영했다


In [6]:
status = run(["git", "status", "--porcelain"], check=False).stdout.strip()
if status:
    run(["git", "commit", "-m", "Track mydata.csv v1 with DVC"], check=True)
else:
    print("No changes to commit.")


$ git status --porcelain
M  .dvc/config
M  data/.gitignore
A  data/mydata.csv.dvc
?? data_versioning_pracitce/

$ git commit -m 'Track mydata.csv v1 with DVC'
[main 43ac0ea] Track mydata.csv v1 with DVC
 3 files changed, 8 insertions(+)
 create mode 100644 data/mydata.csv.dvc



## 6) 현재 DVC 추적 결과 확인

여기서 확인할 수 있는 것:
- `.dvc` 파일이 실제로 생성되었는가
- `dvc remote list` 결과가 보이는가
- GitHub 저장소 링크 제출이 가능한 상태인가


In [7]:
print("Does .dvc file exist? ->", DVC_FILE.exists())
if DVC_FILE.exists():
    print("\n--- Contents of data/mydata.csv.dvc ---")
    print(DVC_FILE.read_text())

print("\n--- DVC remotes ---")
run(["dvc", "remote", "list"], check=False)

print("\n--- GitHub origin URL ---")
run(["git", "remote", "get-url", "origin"], check=False)

print("\n--- Push dataset contents to practice_local remote ---")
run(["dvc", "push", "-r", REMOTE_NAME], check=True)


Does .dvc file exist? -> True

--- Contents of data/mydata.csv.dvc ---
outs:
- md5: bdbf318e67d33982177c171893485143
  size: 110
  hash: md5
  path: mydata.csv


--- DVC remotes ---
$ dvc remote list
localstore      /Users/chaeyooneom/dvcstore/mlops_assignment    (default)
practice_local  /Users/chaeyooneom/dvcstore/mlops_assignment_practice


--- GitHub origin URL ---
$ git remote get-url origin
https://github.com/ChaeYoonUm/mlops_assignment.git


--- Push dataset contents to practice_local remote ---
$ dvc push -r practice_local
2 files pushed



CompletedProcess(args=['dvc', 'push', '-r', 'practice_local'], returncode=0, stdout='2 files pushed\n', stderr='')

## 7) 데이터 수정: 변경 전/후 차이 만들기

이제 같은 CSV를 다시 수정해 **새 버전의 데이터**로 만듭니다.

이번에는 다음과 같은 의미 있는 변경을 넣습니다.

- `Bob`의 나이를 29 → 30으로 수정
- `Charlie`의 구매 금액을 15.0 → 99.9로 수정
- 새 사용자 `Diana` 행 추가


In [8]:
df_before = pd.read_csv(CSV_FILE)

df_v2 = df_before.copy()
df_v2.loc[df_v2["name"] == "Bob", "age"] = 30
df_v2.loc[df_v2["name"] == "Charlie", "purchase_amount"] = 99.9
df_v2 = pd.concat(
    [
        df_v2,
        pd.DataFrame(
            [
                {
                    "user_id": 104,
                    "name": "Diana",
                    "age": 27,
                    "city": "Incheon",
                    "purchase_amount": 55.5,
                }
            ]
        ),
    ],
    ignore_index=True,
)

df_v2.to_csv(CSV_FILE, index=False)

display(Markdown("### 변경 전 데이터"))
display(df_before)

display(Markdown("### 변경 후 데이터"))
display(df_v2)


### 변경 전 데이터

,user_id,name,age,city,purchase_amount
0,101,Alice,25,Suwon,19.5
1,102,Bob,29,Seoul,42.0
2,103,Charlie,31,Busan,15.0


### 변경 후 데이터

,user_id,name,age,city,purchase_amount
0,101,Alice,25,Suwon,19.5
1,102,Bob,30,Seoul,42.0
2,103,Charlie,31,Busan,99.9
3,104,Diana,27,Incheon,55.5


## 8) 수정된 데이터를 다시 DVC로 추적하고, 두 번째 의미 있는 commit 생성

이 단계의 의미:
- 같은 파일이라도 **내용이 바뀌면 새로운 데이터 버전**으로 취급
- `.dvc` 메타파일이 새 해시를 가리키도록 갱신
- Git commit으로 “언제 어떤 데이터 버전으로 바뀌었는지” 기록


In [9]:
run(["dvc", "add", str(CSV_FILE.relative_to(repo_root))], check=True)

to_add = []
for p in [DATA_DIR / ".gitignore", DVC_FILE]:
    if p.exists():
        to_add.append(str(p.relative_to(repo_root)))

run(["git", "add", *to_add], check=True)
run(["git", "status", "--short"], check=False)

status = run(["git", "status", "--porcelain"], check=False).stdout.strip()
if status:
    run(["git", "commit", "-m", "Update mydata.csv to v2 with DVC"], check=True)
else:
    print("No changes to commit.")

run(["dvc", "push", "-r", REMOTE_NAME], check=True)


$ dvc add data/mydata.csv

To track the changes with git, run:

	git add data/mydata.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true

⠋ Checking graph


$ git add data/.gitignore data/mydata.csv.dvc
$ git status --short
M  data/mydata.csv.dvc
?? data_versioning_pracitce/

$ git status --porcelain
M  data/mydata.csv.dvc
?? data_versioning_pracitce/

$ git commit -m 'Update mydata.csv to v2 with DVC'
[main 009507d] Update mydata.csv to v2 with DVC
 1 file changed, 2 insertions(+), 2 deletions(-)

$ dvc push -r practice_local
1 file pushed



CompletedProcess(args=['dvc', 'push', '-r', 'practice_local'], returncode=0, stdout='1 file pushed\n', stderr='')

## 최종 정리

In [10]:
print("1) GitHub 저장소 링크 제출 가능 여부")
run(["git", "remote", "get-url", "origin"], check=False)

print("\n2) README.md 존재 여부")
print((repo_root / "README.md").exists())

print("\n3) 최소 2개 이상의 의미 있는 commit 존재 여부")
run(["git", "log", "--oneline", "-n", "5"], check=False)
run(["git", "rev-list", "--count", "HEAD"], check=False)

print("\n4) .dvc 파일 존재 여부")
print(DVC_FILE.exists())
if DVC_FILE.exists():
    print(DVC_FILE.read_text())

print("\n5) dvc remote list 결과")
run(["dvc", "remote", "list"], check=False)

print("\n6) 최근 commit 사이의 DVC 메타데이터 차이")
run(["git", "diff", "HEAD~1", "HEAD", "--", str(DVC_FILE.relative_to(repo_root))], check=False)


1) GitHub 저장소 링크 제출 가능 여부
$ git remote get-url origin
https://github.com/ChaeYoonUm/mlops_assignment.git


2) README.md 존재 여부
True

3) 최소 2개 이상의 의미 있는 commit 존재 여부
$ git log --oneline -n 5
009507d Update mydata.csv to v2 with DVC
43ac0ea Track mydata.csv v1 with DVC
ac8b58f Track titanic.csv with DVC
464b893 Track titanic.csv with DVC
6c16df0 re-order foldres

$ git rev-list --count HEAD
13


4) .dvc 파일 존재 여부
True
outs:
- md5: b156462ef49683610638c2f9f798e2d9
  size: 136
  hash: md5
  path: mydata.csv


5) dvc remote list 결과
$ dvc remote list
localstore      /Users/chaeyooneom/dvcstore/mlops_assignment    (default)
practice_local  /Users/chaeyooneom/dvcstore/mlops_assignment_practice


6) 최근 commit 사이의 DVC 메타데이터 차이
$ git diff 'HEAD~1' HEAD -- data/mydata.csv.dvc
diff --git a/data/mydata.csv.dvc b/data/mydata.csv.dvc
index 7c52413..8a13171 100644
--- a/data/mydata.csv.dvc
+++ b/data/mydata.csv.dvc
@@ -1,5 +1,5 @@
 outs:
-- md5: bdbf318e67d33982177c171893485143
-  size: 110
+- md5: b1564

CompletedProcess(args=['git', 'diff', 'HEAD~1', 'HEAD', '--', 'data/mydata.csv.dvc'], returncode=0, stdout='diff --git a/data/mydata.csv.dvc b/data/mydata.csv.dvc\nindex 7c52413..8a13171 100644\n--- a/data/mydata.csv.dvc\n+++ b/data/mydata.csv.dvc\n@@ -1,5 +1,5 @@\n outs:\n-- md5: bdbf318e67d33982177c171893485143\n-  size: 110\n+- md5: b156462ef49683610638c2f9f798e2d9\n+  size: 136\n   hash: md5\n   path: mydata.csv\n', stderr='')

## 10) 왜 Git만으로는 부족한가?

### 핵심 이유
1. **대용량 데이터에 비효율적**
   - Git은 코드와 텍스트 파일 버전 관리에는 강하지만, 큰 CSV/Parquet/모델 파일의 잦은 변경 관리에는 비효율적이다.
   - 데이터 버전이 바뀔 때마다 저장소 크기가 빠르게 커질 수 있다.

2. **데이터 버전과 코드 버전의 연결이 약해질 수 있음**
   - 예를 들어 `train.py`는 같아도 어떤 데이터셋 버전으로 학습했는지 Git만으로는 체계적으로 추적하기 어렵다.
   - DVC는 `.dvc` 파일과 해시를 통해 **“이 커밋은 어떤 데이터 버전을 가리키는가”** 를 명시적으로 남긴다.

3. **실제 데이터와 메타데이터를 분리할 수 있음**
   - GitHub에는 코드와 `.dvc` 파일만 올리고,
   - 실제 데이터는 DVC remote(shared storage, NAS, S3, SSH 서버 등)에 저장할 수 있다.
   - 이 구조 덕분에 Git 저장소는 비교적 가볍게 유지된다.

4. **재현성과 협업에 유리**
   - 회사나 연구실에서 팀원이 `git checkout <commit>` 후 `dvc pull`을 실행하면,
   - 그 commit이 가리키는 정확한 데이터 버전을 다시 받아와 같은 실험 환경을 맞출 수 있다.

### 정리
> Git은 코드 버전 관리에 최적화되어 있고 DVC는 데이터/모델 버전 관리에 최적화되어 있다고 볼 수 있다.
